In [1]:
import numpy as np
import shap
import matplotlib.pyplot as plt
import tensorflow as tf
from utils.data_loader import load_as_maps
from utils.ocean_basins import get_region_mask, get_region
from joblib import load
from utils.plot_map import plot_map2

/Users/jakobmeggendorfer/Documents/CAU/Masterarbeit/master-thesis/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def get_gradient_shap_pixel_unet(model, X, i, j,
                                 feature_names=None,
                                 max_samples=5,
                                 background_size=5):
    """
    Compute Gradient SHAP values for a single pixel (i,j) of a full-size U-Net.
    """

    T = X.shape[0]
    rng = np.random.default_rng(42)
    all_idx = np.arange(T)
    rng.shuffle(all_idx)

    n_bg = min(background_size, T // 2)
    n_eval = min(max_samples, T - n_bg)
    idx_bg = all_idx[:n_bg]
    idx_eval = all_idx[n_bg:n_bg + n_eval]

    background_maps = X[idx_bg]
    X_eval = X[idx_eval]

    # ✅ Functional pixel wrapper (keeps model.inputs/outputs accessible)
    pixel_output = tf.keras.layers.Lambda(lambda x: model(x)[:, i, j, 0], name="pixel_output")(model.input)
    pixel_model = tf.keras.Model(inputs=model.input, outputs=pixel_output)

    # ✅ Create SHAP explainer
    explainer = shap.GradientExplainer(pixel_model, background_maps)
    shap_values = explainer.shap_values(X_eval)

    if isinstance(shap_values, list):
        shap_values = shap_values[0]  # (n_eval, 167, 360, F)

    # --- Aggregate per-feature SHAP for that pixel ---
    shap_mean_pixel = np.mean([np.abs(s[i, j, :]) for s in shap_values], axis=0)

    shap_exp = shap.Explanation(
        values=shap_mean_pixel,
        data=np.mean(X_eval[:, i, j, :], axis=0),
        feature_names=feature_names
    )

    return shap_exp, shap_values

In [23]:
model_path = "../../outputs/u-net/best"    
dataset_id = "exp1"

# load model
model = tf.keras.models.load_model(model_path + "/model.keras")

# load and prepare data
X_test, Y_test = load_as_maps(start_year=2014, end_year=2014, datasets=[dataset_id])
map_mask = X_test[0,:,:, 10] == 1

scaler = load(model_path + '/scaler.pkl')
n_samples, h, w, n_features = X_test.shape
X_test_flat = X_test.reshape(-1,n_features)
X_test_scaled_flat = scaler.transform(X_test_flat)
X_test = X_test_scaled_flat.reshape(n_samples, h, w, n_features)

In [ ]:
feature_names = ['SST', 'SAL', 'ice_frac', 'mixed_layer_depth', 'heat_flux_down', 'water_flux_up', 'stress_X', 'stress_Y', 'currents_X', 'currents_Y','tmask','month_sin','month_cos']

shap_exp, shap_maps = get_gradient_shap_pixel_unet(
    model, X_test, 120, 140, feature_names=feature_names, max_samples=5
)


In [40]:
for feature in feature_names:
    data = shap_maps[0, :, :, feature_names.index(feature)].squeeze(-1)
    plot_map(data=data, folder_path="../../outputs/u-net-shap/120-120", title='', file_name=feature, vmin=-0.05, vmax=0.05, cmap='coolwarm')